# Minimum Teacher Investment + DCT Sweep

**Two questions in one eval:**
1. How little teacher training produces useful spectra?
2. How few DCT coefficients (bytes) are needed?

**Design:**
- Strict 70/15/15 Train/Val/Test partition — no data leakage
- Teacher sweep: {50, 100, 200, 500, 1000, 2000} steps
- DCT sweep: {2, 4, 8, 16} coefficients = {16, 32, 128, 256} bytes
- Students use spectral-only Marathon (no directions = pure Tier 1)
- All eval on held-out Test set

**Output:** Teacher steps × DCT coefficients → student speedup matrix

In [ ]:
# Cell 1: Setup + strict data partition
import os, subprocess, numpy as np, shutil
if os.path.exists('/content/nanogpt-prism'):
    subprocess.run(['rm', '-rf', '/content/nanogpt-prism'])
!git clone https://github.com/realityinspector/nanogpt-prism.git /content/nanogpt-prism
%cd /content/nanogpt-prism
!pip install -q transformers tiktoken datasets
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
!python data/shakespeare_char/prepare.py 2>/dev/null

# Strict 70/15/15 partition
data = np.array(np.memmap('data/shakespeare_char/train.bin', dtype=np.uint16, mode='r'))
val_orig = np.array(np.memmap('data/shakespeare_char/val.bin', dtype=np.uint16, mode='r'))
all_data = np.concatenate([data, val_orig])
np.random.seed(42)
np.random.shuffle(all_data)
n = len(all_data)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

for name, start, end in [('shakespeare_eval', 0, train_end),
                          ('shakespeare_teacher', 0, train_end)]:
    d = f'data/{name}'
    os.makedirs(d, exist_ok=True)
    all_data[start:end].astype(np.uint16).tofile(os.path.join(d, 'train.bin'))
    shutil.copy('data/shakespeare_char/meta.pkl', os.path.join(d, 'meta.pkl'))

# Eval dataset uses TEST partition for validation
all_data[val_end:].astype(np.uint16).tofile('data/shakespeare_eval/val.bin')
# Teacher dataset uses VAL partition for validation
all_data[train_end:val_end].astype(np.uint16).tofile('data/shakespeare_teacher/val.bin')

print(f'Train: {train_end:,} | Val: {val_end-train_end:,} | Test: {n-val_end:,}')
print('Ready.')

In [ ]:
# Cell 2: Train teachers + extract spectra at multiple n_dct levels
import subprocess, os, re

teacher_steps_list = [50, 100, 200, 500, 1000, 2000]
n_dct_list = [2, 4, 8, 16]

# Train each teacher once (checkpoint is reused across n_dct extractions)
for steps in teacher_steps_list:
    ckpt = f'out-teacher-{steps}/ckpt.pt'
    if os.path.exists(ckpt):
        print(f'  Teacher {steps}: checkpoint exists.')
        continue
    print(f'  Training teacher ({steps} steps)...')
    result = subprocess.run([
        'python', 'train.py', 'config/train_shakespeare_char.py',
        '--dataset=shakespeare_teacher',
        f'--max_iters={steps}', f'--eval_interval={steps}',
        '--eval_iters=50', f'--log_interval={steps}',
        f'--out_dir=out-teacher-{steps}',
        '--always_save_checkpoint=True', '--compile=False',
        '--prism_init=False', '--wandb_log=False',
    ], capture_output=True, text=True, timeout=600)
    for line in result.stdout.split('\n'):
        m = re.search(r'step \d+: train loss ([\d.]+), val loss ([\d.]+)', line)
        if m: print(f'    val loss: {m.group(2)}')

# Extract spectra at each n_dct for each teacher
for steps in teacher_steps_list:
    for n_dct in n_dct_list:
        cache = f'.prism_cache/t{steps}_d{n_dct}'
        if os.path.exists(f'{cache}/spectra.json'):
            continue
        subprocess.run([
            'python', 'prism_extract.py',
            '--ckpt', f'out-teacher-{steps}/ckpt.pt',
            '--out', cache,
            f'--n_dct={n_dct}',
        ], capture_output=True, timeout=120)
        size = os.path.getsize(f'{cache}/spectra.json') if os.path.exists(f'{cache}/spectra.json') else 0
        n_groups = 5  # attention, attn_proj, ffn_up, ffn_down, embedding
        raw_bytes = n_dct * n_groups * 4
        print(f'  t{steps}_d{n_dct}: {raw_bytes} bytes ({n_dct} coeffs × {n_groups} groups × 4B)')

print('\nAll teachers trained and spectra extracted.')

In [ ]:
# Cell 3: Run student training for every combination
import subprocess, time, re, json, os

STEPS = 5000
EVAL = 100

configs = [('baseline', ['--prism_init=False'])]

# Teacher steps × n_dct matrix (spectral only, no directions)
for teacher_steps in [50, 100, 200, 500, 1000, 2000]:
    for n_dct in [2, 4, 8, 16]:
        name = f't{teacher_steps}_d{n_dct}'
        configs.append((name, [
            '--prism_init=True', '--prism_align=0',
            f'--prism_spectra=.prism_cache/{name}/spectra.json',
            '--learning_rate=5e-4', '--warmup_iters=50',
            '--prism_mod=0.01', '--prism_mod_decay=0.9999',
        ]))

# Full Tier 2 comparison (best teacher + directions)
configs.append(('full_t2000', [
    '--prism_init=True', '--prism_align=0.75',
    '--prism_spectra=.prism_cache/t2000_d8/spectra.json',
    '--prism_directions=.prism_cache/t2000_d8/directions.pt',
    '--learning_rate=5e-4', '--warmup_iters=50',
    '--prism_mod=0.01', '--prism_mod_decay=0.9999',
]))

all_curves = {}

for name, extra in configs:
    log_path = f'/content/mt_{name}.txt'
    if os.path.exists(log_path):
        print(f'[{name}] SKIP')
        with open(log_path) as f:
            stdout = f.read()
    else:
        print(f'  {name}...')
        t0 = time.time()
        result = subprocess.run(
            ['python', 'train.py', 'config/train_shakespeare_char.py',
             '--dataset=shakespeare_eval',
             f'--max_iters={STEPS}', f'--eval_interval={EVAL}',
             '--eval_iters=50', '--log_interval=500',
             f'--out_dir=out-mt-{name}',
             '--wandb_log=False', '--compile=False'] + extra,
            capture_output=True, text=True, timeout=1200
        )
        stdout = result.stdout
        with open(log_path, 'w') as f:
            f.write(stdout)
        if result.returncode != 0:
            print(f'    ERROR: {result.stderr[-300:]}')
        print(f'    {time.time()-t0:.0f}s')

    val = {}
    for line in stdout.split('\n'):
        m = re.search(r'step (\d+): train loss ([\d.]+), val loss ([\d.]+)', line)
        if m:
            val[int(m.group(1))] = float(m.group(3))
    all_curves[name] = val
    if val:
        print(f'  [{name}] best: {min(val.values()):.4f}')

with open('/content/mt_curves.json', 'w') as f:
    json.dump({k: {str(s): v for s, v in c.items()} for k, c in all_curves.items()}, f, indent=2)
print('\nAll saved.')

In [ ]:
# Cell 4: Results matrix
import json

curves = {k: {int(s): v for s, v in c.items()}
          for k, c in json.load(open('/content/mt_curves.json')).items()}

bb = min(curves['baseline'].values())
bs = min(curves['baseline'], key=curves['baseline'].get)

print('='*75)
print('  TEACHER STEPS × DCT COEFFICIENTS → STUDENT SPEEDUP')
print('='*75)
print(f'  Baseline best: {bb:.4f} at step {bs} (on held-out test set)')
print(f'  All students: spectral-only Marathon, no directions')

# Speedup matrix
print(f'\n  {"":>12s}', end='')
for n_dct in [2, 4, 8, 16]:
    raw_bytes = n_dct * 5 * 4
    print(f'  {"d="+str(n_dct)+" ("+str(raw_bytes)+"B)":>14s}', end='')
print()
print(f'  {"-"*72}')

for teacher_steps in [50, 100, 200, 500, 1000, 2000]:
    print(f'  {str(teacher_steps)+" steps":>12s}', end='')
    for n_dct in [2, 4, 8, 16]:
        name = f't{teacher_steps}_d{n_dct}'
        c = curves.get(name, {})
        if not c:
            print(f'  {"—":>14s}', end='')
            continue
        hit = None
        for s in sorted(c.keys()):
            if c[s] <= bb:
                hit = s
                break
        if hit:
            spd = f'{bs/hit:.1f}x'
            print(f'  {spd:>14s}', end='')
        else:
            best = min(c.values())
            print(f'  {"(" + f"{best:.3f}" + ")":>14s}', end='')
    print()

# Full Tier 2 for comparison
c = curves.get('full_t2000', {})
if c:
    hit = None
    for s in sorted(c.keys()):
        if c[s] <= bb:
            hit = s
            break
    spd = f'{bs/hit:.1f}x' if hit else '—'
    print(f'\n  {"Full Tier 2":>12s}  (2000 steps + directions + 500MB): {spd}')

# Best val loss matrix
print(f'\n  --- Best val loss achieved ---')
print(f'  {"":>12s}', end='')
for n_dct in [2, 4, 8, 16]:
    print(f'  {"d="+str(n_dct):>14s}', end='')
print()
print(f'  {"-"*72}')
for teacher_steps in [50, 100, 200, 500, 1000, 2000]:
    print(f'  {str(teacher_steps)+" steps":>12s}', end='')
    for n_dct in [2, 4, 8, 16]:
        name = f't{teacher_steps}_d{n_dct}'
        c = curves.get(name, {})
        if c:
            best = min(c.values())
            marker = ' *' if best < bb else '  '
            print(f'  {best:>12.4f}{marker}', end='')
        else:
            print(f'  {"—":>14s}', end='')
    print()
print(f'  {"Baseline":>12s}  {bb:.4f}')

print(f'\n  * = better than baseline best. Parentheses = never reached baseline.')

In [ ]:
# Cell 5: Download
from google.colab import files
files.download('/content/mt_curves.json')